# 05 - Auto Loader: incremental runs, rescue, and schema evolution

Prerequisites / execution context: Run 00. Uses AvailableNow for serverless compatibility. Creates isolated paths and tables; no cloud notification setup required for the directory-listing exercise.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R10: Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader); [R11: Auto Loader schema inference and evolution](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema); [R12: Auto Loader file detection modes](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-detection-modes); [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


## 1. Ingest initial JSON with a persistent checkpoint
Keep this run_id and both state locations unchanged for all cells in this experiment.


In [ ]:
import uuid
run_id=uuid.uuid4().hex[:10]
base=f"{VOLUME}/auto_{run_id}"
source=f"{base}/input"; schema_path=f"{base}/schema"; checkpoint=f"{base}/checkpoint"
T=f"{BRONZE}.auto_{run_id}"
dbutils.fs.put(f"{source}/a.json",'{"id":"1","amount":"10"}',True)
def ingest():
    df=(spark.readStream.format("cloudFiles").option("cloudFiles.format","json")
        .option("cloudFiles.schemaLocation",schema_path)
        .option("cloudFiles.schemaEvolutionMode","rescue")
        .option("rescuedDataColumn","_rescued_data").load(source))
    q=(df.writeStream.option("checkpointLocation",checkpoint)
       .trigger(availableNow=True).toTable(T))
    q.awaitTermination()
    return q
q=ingest()
assert spark.table(T).count()==1
q=ingest()
assert spark.table(T).count()==1


## 2. Add a column and inspect rescued data
Because rescue mode was selected, the new currency field should be captured in _rescued_data, not added to the inferred schema. Source-schema evolution and target-table schema evolution are separate controls.


In [ ]:
dbutils.fs.put(f"{source}/b.json",'{"id":"2","amount":"20","currency":"EUR"}',True)
q=ingest()
result=spark.table(T)
assert result.count()==2
assert "currency" not in result.columns
assert result.filter("_rescued_data IS NOT NULL").count()==1
result.show(truncate=False)
print(q.lastProgress)


## 3. Optional controlled schema-evolution experiment
Use a separate new path, schema directory, checkpoint and target. First ingest a file containing only id. Then add a file containing id and currency. Configure addNewColumns without supplying a fixed schema; enable mergeSchema on the sink. Expect the first run after the new field to fail after updating the inferred schema. Restart with the SAME state locations; expect currency to become a column. Do not turn an arbitrary production exception into an unconditional retry loop. Official Databricks documentation: [R11: Auto Loader schema inference and evolution](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema).


In [ ]:
RUN_EVOLUTION_EXPERIMENT=False
if RUN_EVOLUTION_EXPERIMENT:
    evo=f"{VOLUME}/evo_{uuid.uuid4().hex[:10]}"
    evo_table=f"{BRONZE}.evo_{uuid.uuid4().hex[:10]}"
    dbutils.fs.put(f"{evo}/input/a.json",'{"id":"1"}',True)
    def evolve():
        df=(spark.readStream.format("cloudFiles").option("cloudFiles.format","json")
            .option("cloudFiles.schemaLocation",f"{evo}/schema")
            .option("cloudFiles.schemaEvolutionMode","addNewColumns").load(f"{evo}/input"))
        q=(df.writeStream.option("checkpointLocation",f"{evo}/checkpoint")
            .option("mergeSchema","true").trigger(availableNow=True).toTable(evo_table))
        q.awaitTermination()
    evolve()
    dbutils.fs.put(f"{evo}/input/b.json",'{"id":"2","currency":"EUR"}',True)
    # Expected to fail once when the new column is discovered.
    # After inspecting that specific error, run evolve() in a NEW cell.
    evolve()


## Explain before moving on
The schema directory tracks the reader schema; the checkpoint tracks progress. Deleting only the checkpoint can replay old files into an existing target. File discovery mode is separate from AvailableNow, which controls query execution. For production, evaluate managed file events on a supported external location. Auto Loader does not promise file order. Official Databricks documentation: [R12: Auto Loader file detection modes](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-detection-modes); [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints).
